In [ ]:
import json
import os

from litellm import completion
from tqdm import tqdm

PROMPT_TEMPLATE = """
You are given a structured instruction describing a user's intent.

Your task is to rewrite it as a natural, realistic user message that a real human would send in a conversation (e.g., chat, email, or customer support request).

Core requirements:

The resulting message MUST:

- Include ALL personal information and contextual details explicitly present in the instruction (e.g., name, ID, order number, location, role, or any identity-related data)
- Start with a self-introduction that fully reflects the user's identity from the instruction
- Contain all requests, conditions, preferences, and constraints without omission
- Be fully sufficient so that an assistant can understand and act without asking follow-up questions
- Be natural, clear, and realistic as a human-written message
- Self-introduction requirement (STRICT):
- The message MUST begin with a self-introduction containing all identity-relevant information from the instruction: e.g. name, order ID, location, role, or any contextual identity provided
- The introduction must be natural and conversational, not robotic or templated
- Do NOT mention "instruction", "structured data", or any meta references

Rules:

- Preserve all meaning, constraints, and details
- Do NOT omit any information, condition, or preference
- Do NOT add new requirements or assumptions
- Do NOT infer missing data
- Do NOT use meta language (e.g., "you are", "the user is", "instruction says")
- Convert everything into a first-person natural message
- Make it sound like a real human writing to get help
- Light conversational phrasing is allowed if it improves readability
- Ensure the message is actionable, complete, and unambiguous

Output format:

Return only the final user message. No explanations.

Examples:

Example 1

Instruction:
You are Alex Chen in order #A123. You want to return a laptop due to overheating and request a refund.

Output:
Hi, I’m Alex Chen and I’m referring to order #A123.

I’d like to request a return for the laptop from this order because it has been overheating frequently. I would also like to request a refund for this purchase.

Example 2

Instruction:
You are Maria Lopez, account ID 88901. You want to change your shipping address and update your subscription plan to premium.

Output:
Hi, I’m Maria Lopez, account ID 88901.

I’d like to update my shipping address for my account and also upgrade my current subscription to the premium plan.

Example 3

Instruction:
You are Yusuf Rossi in 19122. You received your order #W2378156 and wish to exchange the mechanical keyboard for a similar one but with clicky switches and the smart thermostat for one compatible with Google Home instead of Apple HomeKit. If there is no keyboard that is clicky, RGB backlight, full size, you'd rather only exchange the thermostat. You are detail-oriented and want to make sure everything is addressed in one go.

Output:
Hi, I’m Yusuf Rossi in 19122 and I recently received my order #W2378156.

I’d like to request an exchange for a couple of items in this order. I want to exchange the mechanical keyboard for a similar model with clicky switches. I also want to exchange the smart thermostat for one that works with Google Home instead of Apple HomeKit.

If there isn’t a keyboard available that has clicky switches, RGB backlighting, and is full size, then I would prefer to only proceed with exchanging the thermostat instead.

I want to make sure everything is handled properly in one request.

Now your turn

Instruction:

{instruction}

Output:
"""

MODEL = "gpt-oss:20b"
OLLAMA_HOST = "http://localhost:11434"
PROVIDER = "ollama"

DIR = r"user_prompts_v2"

os.makedirs(DIR, exist_ok=True)

with open("instructions_test.json", encoding="utf-8") as f:
    instructions = json.load(f)
print(len(instructions))

for i, instruction in enumerate(tqdm(instructions, desc="Generating prompts")):
    prompt = PROMPT_TEMPLATE.format(instruction=instruction)

    response = completion(
        model=MODEL,
        custom_llm_provider=PROVIDER,
        messages=[{"role": "system", "content": prompt}],
        base_url=OLLAMA_HOST,
        stream=True,
    )

    full_text = ""

    for chunk in response:
        try:
            delta = chunk.get("choices", [{}])[0].get("delta", {})
            token = delta.get("content")

            if token:
                print(token, flush=True, end="")
                full_text += token

        except Exception:
            pass

    filename = f"task_{i}.txt"
    filepath = os.path.join(DIR, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(full_text)

115


Generating prompts:   0%|          | 0/115 [00:00<?, ?it/s]

Hi, I’m Yusuf Rossi in 19122 and I recently received my order #W2378156.

I’d like to exchange the mechanical keyboard for a similar model that has clicky switches, is full size, and has RGB backlighting. If no keyboard is available that meets all those specs, I would be fine with one that has clicky switches but no RGB backlight.

I also want to exchange the smart thermostat for a unit that is compatible with Google Home instead of Apple HomeKit.

Please make sure both of these exchanges are processed together in a single request, as I’d like everything to be handled at once. Thank you for your help.

Generating prompts:   1%|          | 1/115 [00:17<32:54, 17.32s/it]

Hi, I’m Yusuf Rossi in 19122 and I recently received my order #W2378156.

I’d like to request an exchange for a couple of items in this order. I want to exchange the mechanical keyboard for a similar model with clicky switches. I also want to exchange the smart thermostat for one that works with Google Home instead of Apple HomeKit.

If there isn’t a keyboard available that has clicky switches, RGB backlighting, and is full size, then I would prefer to only proceed with exchanging the thermostat instead.

I want to make sure everything is handled properly in one request.

...

Generating prompts:  98%|█████████▊| 113/115 [30:16<00:32, 16.30s/it]

Hi, I’m Yara Silva from zip code 77159.

I’m a bit sad and cautious about making changes, so I’ll break it down piece by piece.

1. I’d like to update the shipping address for my laptop order to my NYC address that’s already in my profile—no need to share the details here.

2. I also want to change the laptop to model number 9844888101.

3. Lastly, I’d like to exchange my watch for one with a black dial but keep the leather strap.

Generating prompts:  99%|█████████▉| 114/115 [30:32<00:16, 16.24s/it]

Hi, I’m Yara Muller and my zip code is 85041.

I’d like to cancel all of my pending orders. I’d prefer not to share a reason right now; if you ask, I’ll say it was ordered by mistake.

Generating prompts: 100%|██████████| 115/115 [30:45<00:00, 16.05s/it]


In [ ]:
import json
import os
import uuid

from litellm import completion
from tqdm import tqdm

FEW_SHOT_EXAMPLE = """
# Example 1: Product Exchange (Simple)

Instruction:
You are Alex Chen. You have order #A12345 and want to exchange wireless headphones for a black color instead of white.

Output:
#Ticket

__metadata__
Ticket ID: #<random_ticket_id>
User Name: Alex Chen  
ZIP Code: 10001  
Email: alex.chen@email.com  

---

__Content__

Hi, I’m Alex Chen and I’m writing regarding order #A12345.

I would like to exchange my wireless headphones for a different color (black instead of white). Everything else is fine, I just prefer a darker color.

Please let me know if this can be processed.

---

# Example 2: Return + Refund

Instruction:
You are Maria Lopez. You want to return a laptop from order #B77891 because it overheats and request a refund.

Output:
#Ticket

__metadata__
Ticket ID: #<random_ticket_id>
User Name: Maria Lopez  
ZIP Code: 33101  
Email: maria.lopez@email.com  

---

__Content__

Hello, I’m Maria Lopez and this is regarding order #B77891.

I want to return the laptop I purchased because it overheats after 10–15 minutes of use. I would like a full refund if possible.

I can provide photos or videos if needed for verification.

---

# Example 3: Multi-item Request + Conditional Fallback

Instruction:
You are Daniel Kim with order #W2378156. You want to exchange a keyboard (clicky switches) and a smart thermostat (Google Home compatible). If no suitable keyboard exists, only proceed with thermostat.

Output:
#Ticket

__metadata__
Ticket ID: #<random_ticket_id>
User Name: Daniel Kim  
ZIP Code: 94107  
Email: daniel.kim@email.com  

---

__Content__

Hi, I’m Daniel Kim and I’m writing regarding order #W2378156.

I would like to request an exchange for two items:

The mechanical keyboard: I want a similar model but with clicky switches.  
The smart thermostat: I want one compatible with Google Home instead of Apple HomeKit.

If there is no keyboard available that meets clicky switches, RGB backlighting, and full-size layout, then please only proceed with the thermostat exchange.

I would prefer everything to be handled in one request to avoid multiple tickets.

---

# Example 4: High-detail / Edge-case

Instruction:
You are Yusuf Rossi with ZIP 90210 and order #W2378156. You want a keyboard (clicky switches, RGB, full-size) and a smart thermostat (Google Home instead of Apple HomeKit). If keyboard unavailable, only thermostat.

Output:
#Ticket

__metadata__
Ticket ID: #<random_ticket_id>
User Name: Yusuf Rossi  
ZIP Code: 90210  
Email: yusuf.rossi@email.com  

---

__Content__

Hi, I’m Yusuf Rossi and I’m writing regarding order #W2378156.

I recently received the order and would like to request an exchange for two items.

First, the mechanical keyboard — I would like a similar model but with clicky switches, and ideally RGB backlighting and full-size layout.

Second, the smart thermostat — I need a version that supports Google Home instead of Apple HomeKit.

However, if no keyboard is available that meets all three conditions (clicky switches, RGB backlighting, full-size), then I prefer to only proceed with exchanging the thermostat.

Please ensure everything is handled in a single resolution if possible.

---

# Example 5: Guest Case (No Customer Information)

Instruction:
You are a guest customer with no account. You want to exchange a keyboard (clicky switches) and a smart thermostat (Google Home instead of Apple HomeKit). If no suitable keyboard exists, only proceed with thermostat.

Output:
#Ticket

__metadata__
Ticket ID: #<random_ticket_id>
User Name: Guest  
ZIP Code: unknown  
Email: unknown  

---

__Content__

Hi, I recently placed an order, but I do not have an account linked.

I would like assistance with an exchange for two items from my order. The first item is a mechanical keyboard — I want a similar model but with clicky switches. The second item is a smart thermostat — I need one compatible with Google Home instead of Apple HomeKit.

If there is no keyboard available that meets clicky switches, RGB backlighting, and full-size layout, then I would prefer to only proceed with the thermostat exchange.

Please help me process this request even without an account, if possible.
"""


PROMPT_TEMPLATE = """
You are given a structured instruction describing a user's intent.

Your task is to rewrite it as a natural, realistic user ticket that a real human would send in a support ticket

Core requirements:

The resulting tickt MUST:

- Include ALL personal information and contextual details explicitly present in the instruction (e.g., name, ID, order number, location, role, or any identity-related data)
- Start with a self-introduction that fully reflects the user's identity from the instruction
- Contain all requests, conditions, preferences, and constraints without omission
- Be fully sufficient so that an assistant can understand and act without asking follow-up questions
- Be natural, clear, and realistic as a human-written tickt
- Self-introduction requirement (STRICT):
- The tickt MUST begin with a self-introduction containing all identity-relevant information from the instruction: e.g. name, order ID, location, role, or any contextual identity provided
- The introduction must be natural and conversational, not robotic or templated
- Do NOT mention "instruction", "structured data", or any meta references

Rules:

- Preserve all meaning, constraints, and details
- Do NOT omit any information, condition, or preference
- Do NOT add new requirements or assumptions
- Do NOT infer missing data
- Do NOT use meta language (e.g., "you are", "the user is", "instruction says")
- Convert everything into a first-person natural tickt
- Make it sound like a real human writing to get help
- Light conversational phrasing is allowed if it improves readability
- Ensure the tickt is actionable, complete, and unambiguous

Output format:

Return only the final user tickt. No explanations.
Tickets must follow the format below:


Examples:

{few_shot_example}

Now your turn

Instruction:

{instruction}

Output: With Ticket ID #{ticket_id}
"""


MODEL = "gpt-oss:20b"
OLLAMA_HOST = "http://localhost:11434"
PROVIDER = "ollama"

DIR = r"user_tickets"

os.makedirs(DIR, exist_ok=True)

with open("instructions_test.json", encoding="utf-8") as f:
    instructions = json.load(f)
print(len(instructions))

for i, instruction in enumerate(tqdm(instructions, desc="Generating prompts")):
    ticket_id = str(uuid.uuid4())
    prompt = PROMPT_TEMPLATE.format(
        few_shot_example=FEW_SHOT_EXAMPLE, instruction=instruction, ticket_id=ticket_id
    )

    response = completion(
        model=MODEL,
        custom_llm_provider=PROVIDER,
        messages=[{"role": "system", "content": prompt}],
        base_url=OLLAMA_HOST,
        stream=True,
    )

    full_text = ""

    for chunk in response:
        try:
            delta = chunk.get("choices", [{}])[0].get("delta", {})
            token = delta.get("content")

            if token:
                print(token, flush=True, end="")
                full_text += token

        except Exception:
            pass

    filename = f"task_{i}.txt"
    filepath = os.path.join(DIR, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(full_text)

115


Generating prompts:   0%|          | 0/115 [00:00<?, ?it/s]

#Ticket

__metadata__
Ticket ID: #d5be4bf3-e4d2-439d-9f9c-df2115bf3bc9
User Name: Yusuf Rossi  
ZIP Code: 19122  
Email: unknown  

---

__Content__

Hi, I’m Yusuf Rossi and I received my order #W2378156.  

I’d like to exchange two items from that order:

- **Mechanical keyboard** – I need a similar model that has clicky switches. Ideally it would also feature RGB backlighting and be full‑size. If no keyboard meets all three of those criteria, I’m willing to accept a version that has clicky switches but without RGB backlighting.

- **Smart thermostat** – I want a model that is compatible with Google Home instead of Apple HomeKit.

Please let me know if you can process both exchanges in one go, and confirm the new items, shipping details, and any additional steps I need to take.  

Thank you!

Generating prompts:   1%|          | 1/115 [00:20<38:55, 20.49s/it]

#Ticket

__metadata__
Ticket ID: #abb66e97-1bd3-44d0-ab81-1d1d6a156167  
User Name: Yusuf Rossi  
ZIP Code: 19122  
Email: unknown  

---

__Content__

Hi, I’m Yusuf Rossi from ZIP 19122, and I’m writing regarding my recent order #W2378156.

I’d like to request an exchange for two items:

1. **Mechanical keyboard** – I’d like a similar model but with clicky switches, RGB backlighting, and a full‑size layout.  
2. **Smart thermostat** – I need one that is compatible with Google Home instead of Apple HomeKit.

If there is no keyboard available that meets the clicky switches, RGB backlight, and full‑size criteria, I would prefer to proceed only with the thermostat exchange.

I’m detail‑oriented and would appreciate it if everything could be handled in a single resolution to avoid multiple tickets. Thank you!

...

Generating prompts:  98%|█████████▊| 113/115 [34:59<00:41, 20.55s/it]

#Ticket

__metadata__
Ticket ID: #3336c653-8903-4c4b-82a9-4f867d179434
User Name: Yara Silva  
ZIP Code: 77159  
Email: unknown  

---

__Content__

Hi, I’m Yara Silva, and I’m feeling a bit sad and cautious about these changes.

First, I’d like to update my laptop order so that the shipping address is my NYC address. I’m not comfortable sharing the exact address here, but it should already be saved in my orders profile.

Second, I want to modify the laptop itself to the model number 9844888101.

Third, I’d like to exchange my watch for a black dial version, but I’d like to keep the leather strap that it currently has.

I’d appreciate it if you could handle these requests in the order I listed them. Thank you!

Generating prompts:  99%|█████████▉| 114/115 [35:16<00:19, 19.38s/it]

#Ticket

__metadata__
Ticket ID: #39208c00-1b59-46ee-99de-16d863b30317
User Name: Yara Muller  
ZIP Code: 85041  
Email: unknown  

---

__Content__

Hi, I’m Yara Muller and I’d like to cancel all of my pending orders. I’m not ready to give a reason right now; I’ll share it if you ask. If you do ask, I’ll say I ordered by mistake. Thanks.

Generating prompts: 100%|██████████| 115/115 [35:30<00:00, 18.53s/it]
